In [0]:
# Load Bronze tables
from pyspark.sql import functions as F
from pyspark.sql.window import Window

unemp = spark.table("ecommerce.bronze.unemployment_bronze")
jobs  = spark.table("ecommerce.bronze.jobs_bronze")

print(f"Unemployment: {unemp.count():,} rows ✓")
print(f"Jobs: {jobs.count():,} rows ✓")

Unemployment: 267 rows ✓
Jobs: 9,355 rows ✓


In [0]:
# State level unemployment features
state_features = unemp.groupBy("region").agg(
    F.avg("unemployment_rate")
     .alias("avg_unemployment_rate"),
    F.max("unemployment_rate")
     .alias("peak_unemployment_rate"),
    F.min("unemployment_rate")
     .alias("min_unemployment_rate"),
    F.stddev("unemployment_rate")
     .alias("unemployment_volatility"),
    F.avg("labour_participation_rate")
     .alias("avg_labour_participation"),
    F.avg("estimated_employed")
     .alias("avg_employed_count"),
    F.count("*").alias("months_tracked")
).withColumn(
    "avg_unemployment_rate",
    F.round(F.col("avg_unemployment_rate"), 2)
).withColumn(
    "risk_level",
    F.when(F.col("avg_unemployment_rate") >= 20, "High")
     .when(F.col("avg_unemployment_rate") >= 10, "Medium")
     .otherwise("Low")
)

print("State features:")
state_features.orderBy(
    F.desc("avg_unemployment_rate")
).show(10, truncate=False)

State features:
+----------------+---------------------+----------------------+---------------------+-----------------------+------------------------+------------------+--------------+----------+
|region          |avg_unemployment_rate|peak_unemployment_rate|min_unemployment_rate|unemployment_volatility|avg_labour_participation|avg_employed_count|months_tracked|risk_level|
+----------------+---------------------+----------------------+---------------------+-----------------------+------------------------+------------------+--------------+----------+
|Haryana         |27.48                |43.22                 |19.68                |6.8193793298543275     |42.099999999999994      |6844059.0         |10            |High      |
|Tripura         |25.06                |41.23                 |11.57                |8.656337755271181      |57.84800000000001       |1397291.6         |10            |High      |
|Jharkhand       |19.54                |59.23                 |7.63                 

In [0]:
# Job market features
category_demand = jobs.groupBy("job_category").agg(
    F.count("*").alias("total_jobs"),
    F.avg("salary_in_usd").alias("avg_salary_usd"),
    F.countDistinct("job_title").alias("unique_roles")
).orderBy(F.desc("total_jobs"))

exp_demand = jobs.groupBy("experience_level").agg(
    F.count("*").alias("jobs_count"),
    F.avg("salary_in_usd").alias("avg_salary_usd")
).orderBy(F.desc("jobs_count"))

top_roles = jobs.groupBy("job_title", "job_category").agg(
    F.count("*").alias("demand_count"),
    F.avg("salary_in_usd").alias("avg_salary_usd"),
    F.max("salary_in_usd").alias("max_salary_usd")
).orderBy(F.desc("avg_salary_usd"))

print("Job category demand:")
category_demand.show(10, truncate=False)
print("Experience level demand:")
exp_demand.show()
print("Top paying roles:")
top_roles.show(10, truncate=False)

Job category demand:
+------------------------------+----------+------------------+------------+
|job_category                  |total_jobs|avg_salary_usd    |unique_roles|
+------------------------------+----------+------------------+------------+
|Data Science and Research     |3014      |163758.57597876576|23          |
|Data Engineering              |2260      |146197.65619469027|15          |
|Data Analysis                 |1457      |108505.72134522992|14          |
|Machine Learning and AI       |1428      |178925.84733893556|29          |
|Leadership and Management     |503       |145476.0198807157 |14          |
|BI and Visualization          |313       |135092.10223642172|11          |
|Data Architecture and Modeling|259       |156002.35907335908|7           |
|Data Management and Strategy  |61        |103139.9344262295 |5           |
|Data Quality and Operations   |55        |100879.47272727273|6           |
|Cloud and Database            |5         |155000.0          |1    

In [0]:
# Build skill gap table
entry_total = jobs.filter(
    F.col("experience_level") == "Entry-level"
).count()

skill_gap = state_features.withColumn(
    "skill_gap_score",
    F.round(
        F.col("avg_unemployment_rate") *
        (1 - F.col("avg_labour_participation") / 100), 2
    )
).withColumn(
    "entry_job_opportunity", F.lit(entry_total)
).withColumn(
    "intervention_priority",
    F.when(
        (F.col("risk_level") == "High") &
        (F.col("skill_gap_score") > 15), "Critical"
    ).when(
        F.col("risk_level") == "High", "High"
    ).when(
        F.col("risk_level") == "Medium", "Monitor"
    ).otherwise("Stable")
)

print("Skill gap by state:")
skill_gap.select(
    "region",
    "avg_unemployment_rate",
    "skill_gap_score",
    "risk_level",
    "intervention_priority"
).orderBy(F.desc("skill_gap_score")).show(15, truncate=False)

Skill gap by state:
+----------------+---------------------+---------------+----------+---------------------+
|region          |avg_unemployment_rate|skill_gap_score|risk_level|intervention_priority|
+----------------+---------------------+---------------+----------+---------------------+
|Haryana         |27.48                |15.91          |High      |Critical             |
|Bihar           |19.47                |12.23          |Medium    |Monitor              |
|Delhi           |18.41                |11.81          |Medium    |Monitor              |
|Jharkhand       |19.54                |11.65          |Medium    |Monitor              |
|Puducherry      |17.94                |11.5           |Medium    |Monitor              |
|Tripura         |25.06                |10.56          |High      |High                 |
|Jammu & Kashmir |16.48                |10.23          |Medium    |Monitor              |
|Himachal Pradesh|16.07                |9.6            |Medium    |Monitor      

In [0]:
# Save all Silver tables
tables = [
    (state_features,  "state_features_silver"),
    (skill_gap,       "skill_gap_silver"),
    (category_demand, "category_demand_silver"),
    (top_roles,       "top_roles_silver"),
    (exp_demand,      "exp_demand_silver"),
]

for df, name in tables:
    df.write.format("delta").mode("overwrite") \
      .saveAsTable(f"ecommerce.silver.{name}")
    print(f"Saved: ecommerce.silver.{name} ✓")

print("\nSilver layer complete ✓")

Saved: ecommerce.silver.state_features_silver ✓
Saved: ecommerce.silver.skill_gap_silver ✓
Saved: ecommerce.silver.category_demand_silver ✓
Saved: ecommerce.silver.top_roles_silver ✓
Saved: ecommerce.silver.exp_demand_silver ✓

Silver layer complete ✓
